# 10. Validation XML

Le XML est **fragile** : un caractère ou une balise mal placée peut rendre un fichier entier illisible pour l'éditeur et pour vos scripts. Ce chapitre distingue deux problèmes :

1. **XML mal formé** — la syntaxe est cassée (guillemets, fermeture de balises, etc.).
2. **XML invalide au regard du schéma** — la syntaxe est correcte, mais une balise ou un attribut n'est **pas autorisé** par les règles TEI (ou par votre personnalisation) du projet.

Le mode **Visual** n'autorise que ce que le schéma permet. Le mode **Source** laisse taper n'importe quoi, mais **Validation** et la sauvegarde vous rappellent à l'ordre. Si vous lancez un script Python sur le dossier, c'est **à vous** d'ajouter des contrôles — l'éditeur ne protège pas les fichiers modifiés de l'extérieur.

**Prérequis :** vous avez fait une sauvegarde (Time Machine ou copie), comme au ch. 9. Ici, on casse volontairement pour apprendre à réparer.

## Exercice 1 : XML mal formé

1. Ouvrez une transcription et basculez en mode **Source**.
2. Repérez une balise `<p>` et remplacez-la par `<p a>` (attribut sans nom ni valeur valide).

Vous devriez voir immédiatement plusieurs signaux :

- une **icône d'avertissement** orange avec un compteur ;
- du **soulignement rouge** à l'endroit fautif ;
- une **barre rouge** près de l'aperçu ;
- en cliquant sur l'icône, la liste des problèmes dans le panneau **Validation** ;
- à la sauvegarde ou en quittant **Source**, un **numéro de ligne** et un **message** explicite.

3. Corrigez `<p a>` en `<p>` et observez la disparition des alertes.

**Réflexe :** lisez le **premier** message de Validation, pas seulement le dernier — les erreurs en cascade partent souvent d'une seule faute en amont.

Si vous voulez éviter ce cas en production, restez en **Visual** pour l'édition courante, ou faites valider le XML dans tout script (parse avec `lxml` / `xml.etree` et arrêtez-vous à la première exception).

## Exercice 2 : violation du schéma

Vous voulez distinguer personnages **historiques** et **mythiques**. Un attribut `@is_mythic` ou une balise dédiée semble naturel.

1. En mode **Visual**, essayez d'ajouter un attribut ou une balise exotique (par ex. `@has_bananas`, `<bananaMan>`) sur un nom de personne.

L'éditeur **refuse** : votre projet est associé à un **schéma** (souvent TEI ALL ou une variante dans `schema/`), et Visual n'offre que ce qui est déclaré.

2. Passez en **Source** et remplacez par exemple :

   `<persName>張某</persName>` → `<bananaMan has_bananas="1">張某</bananaMan>`

3. Enregistrez ou consultez **Validation** : le fichier peut être **syntaxiquement** correct, mais **invalide** — même type d'alertes que pour le XML mal formé, avec un message qui parle de schéma / d'élément inconnu.

4. Revenez à `<persName>…</persName>` (ou corrigez via Time Machine si vous avez propagé la bêtise).

Le schéma n'est pas une punition : c'est un **contrat** partagé entre vous, l'éditeur, vos collègues et vos outils d'analyse.

## D'abord : le TEI standard suffit-il ?

Avant d'inventer `<bananaMan>`, consultez ce qui existe déjà.

- Le schéma actif est indiqué **en bas de l'écran** (souvent un fichier dans `schema/`, par ex. `tei_all.rng`).
- Pour les noms de personnes, la page TEI de référence [`persName`](https://www.tei-c.org/release/doc/tei-p5-doc/fr/html/ref-persName.html) liste les **attributs autorisés** (`@key`, `@type`, `@cert`, etc.) et **où** l'élément peut apparaître.
- Pour « mythique vs historique », `@type="mythic"` (ou une valeur documentée dans votre projet) sur **`persName`** est souvent **suffisant** — sans toucher au schéma.

Dans LJB, le fichier `schema/tei_all.rng` du projet peut déjà être une **personnalisation** (par ex. extensions pour les dates est-asiatiques), pas le TEI ALL « brut » du site tei-c.org. Avant de remplacer ce fichier, notez ce qu'il contient et versionnez-le.

## Personnaliser le schéma avec Roma

Si le TEI standard ne couvre vraiment pas votre besoin (nouvel élément, nouvel attribut global, contrainte particulière), le Consortium TEI propose l'outil web **[Roma](https://romabeta.tei-c.org/)** pour construire une **personnalisation ODD** et en exporter un schéma Relax NG (`.rng`). Documentation générale : [Customizing the TEI](https://tei-c.org/guidelines/customization/#section-1). Tutoriel pas à pas : [TEI by Example — module 8 (ODD / Roma)](https://teibyexample.org/exist/tutorials/TBED08v00.htm).

Workflow recommandé :

1. **Écrire le besoin** en français clair : nom de l'élément ou attribut, exemples XML valides, exemples **invalides** à refuser, parents autorisés (ex. « dans `<p>` »), attributs nécessaires (`@key` pour la désambiguïsation, etc.).
2. Dans **Roma**, partir du modèle **TEI All** (ou recharger un **ODD** déjà sauvegardé).
3. Ajouter ou modifier l'élément / l'attribut dans l'interface ; vérifier la documentation générée.
4. **Télécharger l'ODD** (fichier source à conserver dans Git) et **générer le Relax NG**.
5. Copier le `.rng` (et la CSS si vous en avez une) dans le dossier **`schema/`** du projet ; mettre à jour le schéma actif dans les réglages du projet si besoin.
6. Rouvrir le document, tester en **Visual** que votre balise apparaît, et contrôler **Validation** sur un extrait.

Pour un élément proche de `persName` (comme `<bananaMan>`), Roma doit déclarer où il peut vivre dans l'arbre TEI et quels attributs il partage (`@key`, `@xml:id`, …). Sans cela, Visual et la désambiguïsation ne le traiteront pas comme les autres noms.

**Limites :** Roma ne fait pas tout ; les cas très complexes demandent parfois d'éditer l'ODD à la main et de passer par les outils TEI (TEIGarage). Gardez toujours **ODD + `.rng` + note de projet** ensemble.

## Ce n'est pas que le schéma : liens entre fichiers

Validation TEI ne dit rien sur :

- un `@key` qui ne correspond à **aucune** fiche dans la base SQLite ;
- une **traduction** orpheline (paragraphe supprimé ou UUID changé) ;
- un fichier **renommé ou déplacé** hors du dossier de projet.

Symptômes : panneaux vides, liens morts, comptages incohérents au ch. 8. **Ne déplacez pas** les fichiers liés sans copie de sécurité ; vérifiez chemins et identifiants avant de paniquer.

## Si une automatisation a tout cassé

Arrêtez le script. **Ne propagez pas** une correction aveugle. Travaillez sur une **copie** ou restaurez un snapshot Time Machine (ch. 9). Reproduisez le problème sur **un seul fichier**, puis élargissez.

## Procédure de récupération (checklist)

1. Ne supprimez pas l'original.
2. Notez ce qui a été fait et quand.
3. Lisez le **premier** message dans **Validation**.
4. Identifiez le dernier état sain (Time Machine, Git, copie).
5. Restaurez ou comparez (diff) avec cet état.
6. Pour demander de l'aide : extrait XML **minimal**, message d'erreur complet, étapes pour reproduire — [issue GitHub](https://github.com/lejeanbaptiste/lejeanbaptiste/issues) si c'est un bug outil.

## Exercices

- [ ] Provoquer puis corriger une erreur de **XML mal formé** en Source.
- [ ] Provoquer une erreur de **schéma**, puis revenir à un `persName` valide.
- [ ] Ouvrir la fiche TEI **`persName`** et noter deux attributs utiles pour **votre** projet.
- [ ] Expliquer en une phrase si `@type="mythic"` suffit, ou pourquoi vous auriez besoin de Roma.
- [ ] (Optionnel) Explorer Roma à partir de TEI All **sans** écraser le `schema/` de production — sur une copie du projet.

## Ressources

- [XML (W3C)](https://www.w3.org/XML/) — syntaxe de base.
- [TEI Guidelines (anglais)](https://tei-c.org/release/doc/tei-p5-doc/en/html/) — éléments et modèles.
- [Roma — personnalisation ODD](https://romabeta.tei-c.org/).
- [XPath (MDN)](https://developer.mozilla.org/fr/docs/Web/XML/XPath) — retrouver une balise fautive dans un gros fichier.

Le chapitre 11 reprend l'**autonomie** : formuler une question, choisir les balises, et rédiger une mini-spécification pour la suite de votre recherche.
